In [ ]:
pip install scikit-posthocs

In [3]:
folder_result_path = r"F:\AGF_IVIF\data\output\sota_excel"
files = [f"t{i}.xlsx" for i in range(1, 7)]
methods = [f"t{i}" for i in range(1, 7)]
metrics = ["MLI", "SD", "EN", "MI", "SSIM", "Qabf"]

### Pipeline 1: Kruskal-Wallis + Dunn's

#### Kruskal-Wallis p-value

In [4]:
import os
import pandas as pd
from scipy.stats import kruskal

# đọc toàn bộ dữ liệu
data = []
for f in files:
    file_path = os.path.join(folder_result_path, f)
    df = pd.read_excel(file_path)
    data.append(df)

# tính Kruskal-Wallis cho từng metric
results = {}

for metric in metrics:
    
    groups = []
    
    for df in data:
        values = df[metric].values.astype(float)
        
        # chuẩn hóa nếu là MLI hoặc SD
        if metric in ["MLI", "SD"]:
            values = values / 255.0
        
        groups.append(values)
    
    # Kruskal-Wallis test
    stat, p_value = kruskal(*groups)
    
    results[metric] = {
        "statistic": stat,
        "p_value": p_value
    }

# in kết quả
print("Kruskal-Wallis Results:")
for metric, res in results.items():
    print(f"{metric}: statistic={res['statistic']:.6f}, p-value={res['p_value']:.6e}")

# Lưu kết quả vào Excel
results_df = pd.DataFrame.from_dict(results, orient="index")
results_df.to_excel(os.path.join(folder_result_path, "kruskal_wallis_results.xlsx"))

Kruskal-Wallis Results:
MLI: statistic=24.542483, p-value=1.707268e-04
SD: statistic=46.639398, p-value=6.729088e-09
EN: statistic=24.418983, p-value=1.803389e-04
MI: statistic=323.188648, p-value=1.031584e-67
SSIM: statistic=297.601516, p-value=3.283174e-62
Qabf: statistic=230.545116, p-value=8.172518e-48


#### Dunn Test p-value

In [5]:
import os
import pandas as pd
import scikit_posthocs as sp

# đọc dữ liệu
data = {}
for f, m in zip(files, methods):
    df = pd.read_excel(os.path.join(folder_result_path, f))
    data[m] = df

# các cặp cần so sánh
pairs = ["t6 vs t1", "t6 vs t2", "t6 vs t3", "t6 vs t4", "t6 vs t5"]

# dataframe kết quả
result_table = pd.DataFrame(index=pairs, columns=metrics)

for metric in metrics:

    rows = []

    for m in methods:
        values = data[m][metric].astype(float)

        if metric in ["MLI", "SD"]:
            values = values / 255.0

        for v in values:
            rows.append([v, m])

    df_long = pd.DataFrame(rows, columns=["value", "method"])

    # Dunn test
    dunn = sp.posthoc_dunn(
        df_long,
        val_col="value",
        group_col="method",
        p_adjust="bonferroni"
    )

    # lấy các p-value t6 so với các phương pháp khác
    result_table.loc["t6 vs t1", metric] = dunn.loc["t6", "t1"]
    result_table.loc["t6 vs t2", metric] = dunn.loc["t6", "t2"]
    result_table.loc["t6 vs t3", metric] = dunn.loc["t6", "t3"]
    result_table.loc["t6 vs t4", metric] = dunn.loc["t6", "t4"]
    result_table.loc["t6 vs t5", metric] = dunn.loc["t6", "t5"]

# chuyển sang float
result_table = result_table.astype(float)

print("\nDunn Test p-value Table\n")
print(result_table)

# lưu ra Excel
output_path = os.path.join(folder_result_path, "dunn_test_results.xlsx")
result_table.to_excel(output_path)

print("\nSaved to:", output_path)


Dunn Test p-value Table

               MLI            SD        EN            MI          SSIM  \
t6 vs t1  0.273518  1.880504e-07  0.001708  2.352420e-24  3.161303e-29   
t6 vs t2  1.000000  2.049078e-03  0.319015  1.612877e-06  1.484472e-03   
t6 vs t3  1.000000  1.000000e+00  1.000000  5.134232e-02  2.600795e-03   
t6 vs t4  0.000131  1.115956e-02  1.000000  2.426959e-28  4.866089e-12   
t6 vs t5  0.939462  3.124675e-01  0.007252  2.198095e-01  7.894232e-10   

                  Qabf  
t6 vs t1  1.022537e-21  
t6 vs t2  5.473763e-04  
t6 vs t3  4.033310e-01  
t6 vs t4  1.205853e-18  
t6 vs t5  1.424588e-05  

Saved to: F:\AGF_IVIF\data\output\sota_excel\dunn_test_results.xlsx


### Friedman + Wilcoxon Signed-Rank

#### Frideman

In [6]:
import os
import pandas as pd
from scipy.stats import friedmanchisquare

# đọc dữ liệu
data = {}
for f, m in zip(files, methods):
    df = pd.read_excel(os.path.join(folder_result_path, f))
    df = df.sort_values("Image")  # đảm bảo cùng thứ tự ảnh
    data[m] = df

# bảng kết quả
result_table = pd.DataFrame(columns=["Statistic", "p_value"], index=metrics)

for metric in metrics:

    groups = []

    for m in methods:
        values = data[m][metric].astype(float)

        # chuẩn hóa
        if metric in ["MLI", "SD"]:
            values = values / 255.0

        groups.append(values)

    stat, p = friedmanchisquare(*groups)

    result_table.loc[metric, "Statistic"] = stat
    result_table.loc[metric, "p_value"] = p

result_table = result_table.astype(float)

print("\nFriedman Test Results\n")
print(result_table)

# lưu Excel
output_path = os.path.join(folder_result_path, "friedman_test_results.xlsx")
result_table.to_excel(output_path)

print("\nSaved to:", output_path)


Friedman Test Results

       Statistic       p_value
MLI   243.469136  1.384120e-50
SD    317.500882  1.726207e-66
EN    298.820106  1.796076e-62
MI    375.680776  5.157826e-79
SSIM  284.809524  1.843380e-59
Qabf  317.268078  1.937183e-66

Saved to: F:\AGF_IVIF\data\output\sota_excel\friedman_test_results.xlsx


#### Wilcoxon Signed-Rank

In [7]:
import os
import pandas as pd
from scipy.stats import wilcoxon

folder_result_path = r"F:\AGF_IVIF\data\output\sota_excel"

files = [f"t{i}.xlsx" for i in range(1, 7)]
methods = [f"t{i}" for i in range(1, 7)]

metrics = ["MLI", "SD", "EN", "MI", "SSIM", "Qabf"]

# đọc dữ liệu
data = {}
for f, m in zip(files, methods):
    df = pd.read_excel(os.path.join(folder_result_path, f))
    data[m] = df.sort_values("Image")   # đảm bảo cùng thứ tự ảnh

pairs = ["t6 vs t1", "t6 vs t2", "t6 vs t3", "t6 vs t4", "t6 vs t5"]

result_table = pd.DataFrame(index=pairs, columns=metrics)

for metric in metrics:

    values_t6 = data["t6"][metric].astype(float)

    if metric in ["MLI", "SD"]:
        values_t6 = values_t6 / 255.0

    for i in range(1, 6):

        values_ti = data[f"t{i}"][metric].astype(float)

        if metric in ["MLI", "SD"]:
            values_ti = values_ti / 255.0

        stat, p = wilcoxon(values_t6, values_ti)

        result_table.loc[f"t6 vs t{i}", metric] = p


result_table = result_table.astype(float)

print("\nWilcoxon Signed-Rank Test p-value Table\n")
print(result_table)

# lưu Excel
output_path = os.path.join(folder_result_path, "wilcoxon_results.xlsx")
result_table.to_excel(output_path)

print("\nSaved to:", output_path)


Wilcoxon Signed-Rank Test p-value Table

                   MLI            SD            EN            MI  \
t6 vs t1  2.373533e-10  5.362626e-15  5.362626e-15  5.362626e-15   
t6 vs t2  1.002606e-09  5.362626e-15  3.147540e-14  5.362626e-15   
t6 vs t3  3.384715e-14  7.466771e-06  1.987392e-11  5.566891e-15   
t6 vs t4  7.501579e-15  4.206755e-14  6.482973e-14  5.362626e-15   
t6 vs t5  8.634445e-14  5.362626e-15  5.362626e-15  1.437948e-11   

                  SSIM          Qabf  
t6 vs t1  5.362626e-15  5.362626e-15  
t6 vs t2  1.330356e-04  2.529830e-14  
t6 vs t3  1.108255e-13  5.566891e-15  
t6 vs t4  5.362626e-15  5.362165e-15  
t6 vs t5  2.303872e-06  2.822110e-14  

Saved to: F:\AGF_IVIF\data\output\sota_excel\wilcoxon_results.xlsx
